In [1]:
import random
import numpy as np
from collections import defaultdict

# A. Setting up the environment
q_table = defaultdict(lambda: np.zeros(9))
alpha = 0.1
gamma = 0.9
epsilon = 1.0

# B. Defining the Tic-Tac-Toe game
def check_winner(board, mark):
    wins = [
        (0, 1, 2), (3, 4, 5), (6, 7, 8),
        (0, 3, 6), (1, 4, 7), (2, 5, 8),
        (0, 4, 8), (2, 4, 6)
    ]

    return any(all(board[i] == mark for i in line) for line in wins)


def available_moves(board):
    return [i for i in range(9) if board[i] == " "]


def show_board(board):
    for i in range(0, 9, 3):
        print(" | ".join(board[i:i+3]))
        if i < 6:
            print("--+---+--")


# C. Building the reinforcement learning model
def choose_action(state, moves, explore=True):
    if explore and random.random() < epsilon:
        return random.choice(moves)

    values = q_table[state]
    best_value = max(values[i] for i in moves)
    best_moves = [i for i in moves if values[i] == best_value]
    return random.choice(best_moves)


def update_q(state, action, reward, next_state=None, next_moves=None):
    old_value = q_table[state][action]

    if next_state is None:
        future_value = 0
    else:
        future_value = max(q_table[next_state][i] for i in next_moves)

    q_table[state][action] = old_value + alpha * (
        reward + gamma * future_value - old_value
    )


# D. Training the model
for episode in range(20000):
    board = [" "] * 9

    while True:
        state = tuple(board)
        moves = available_moves(board)

        # Agent plays X
        action = choose_action(state, moves)
        board[action] = "X"

        if check_winner(board, "X"):
            update_q(state, action, 1)
            break

        if not available_moves(board):
            update_q(state, action, 0)
            break

        # Random opponent plays O
        board[random.choice(available_moves(board))] = "O"

        if check_winner(board, "O"):
            update_q(state, action, -1)
            break

        if not available_moves(board):
            update_q(state, action, 0)
            break

        next_state = tuple(board)
        next_moves = available_moves(board)
        update_q(state, action, 0, next_state, next_moves)

    epsilon = max(0.05, epsilon * 0.9998)

print("Training completed!")


# E. Testing the trained model
wins = 0
losses = 0
draws = 0

for game in range(1000):
    board = [" "] * 9

    while True:
        state = tuple(board)
        moves = available_moves(board)

        # Trained agent plays X
        action = choose_action(state, moves, explore=False)
        board[action] = "X"

        if check_winner(board, "X"):
            wins += 1
            break

        if not available_moves(board):
            draws += 1
            break

        # Random opponent plays O
        board[random.choice(available_moves(board))] = "O"

        if check_winner(board, "O"):
            losses += 1
            break

        if not available_moves(board):
            draws += 1
            break

print("Results after 1000 test games:")
print("Wins:", wins)
print("Losses:", losses)
print("Draws:", draws)
print("Win percentage:", round(wins / 1000 * 100, 2), "%")

Training completed!
Results after 1000 test games:
Wins: 983
Losses: 2
Draws: 15
Win percentage: 98.3 %
